In [2]:
import os, requests, pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
headers = {"Authorization": f"Bearer {os.environ['CFBD_KEY']}"}

In [3]:
url = "https://api.collegefootballdata.com/ppa/players/season"
params = {"year": 2026, "position": "QB", "excludeGarbageTime": "true"}
r = requests.get(url, params=params, headers=headers)
r.raise_for_status()

df = pd.json_normalize(r.json())
df.to_csv("../data/qb_ppa_raw.csv", index=False)

In [4]:
#get PPA

qbs = [
    "Darian Mensah", "Dante Moore", "Trinidad Chambliss", "Sam Leavitt",
    "Demond Williams Jr.", "Josh Hoover", "Drew Mestemaker",
    "Jaron-Keawe Sagapolutele", "Mason Heintschel", "Kevin Jennings",
]

out = df[df["name"].isin(qbs)].drop_duplicates(subset="id")
print(set(qbs) - set(out["name"]))
out[["name", "team", "averagePPA.all", "averagePPA.pass"]]

set()


,name,team,averagePPA.all,averagePPA.pass
19,Josh Hoover,Indiana,0.706,0.699
87,Dante Moore,Oregon,0.419,0.433
110,Trinidad Chambliss,Ole Miss,0.451,0.471
144,Sam Leavitt,LSU,0.376,0.396
159,Demond Williams Jr.,Washington,0.242,0.236
178,Kevin Jennings,SMU,0.382,0.513
219,Darian Mensah,Miami,0.727,0.705
340,Jaron-Keawe Sagapolutele,California,0.077,0.140
378,Mason Heintschel,Pittsburgh,0.401,0.404
393,Drew Mestemaker,Oklahoma State,0.355,0.387


In [5]:
# get plays

url = "https://api.collegefootballdata.com/stats/player/season"
r = requests.get(url, params={"year": 2026}, headers=headers)
r.raise_for_status()

stats = pd.json_normalize(r.json())
stats.to_csv("../data/player_stats_raw.csv", index=False)
stats.columns

Index(['season', 'playerId', 'player', 'position', 'team', 'conference',
       'category', 'statType', 'stat'],
      dtype='str')

In [6]:
qb_stats = stats[stats["player"].isin(qbs)].copy()
qb_stats["stat"] = pd.to_numeric(qb_stats["stat"], errors="coerce")

attempts = qb_stats[
    ((qb_stats["category"] == "passing") & (qb_stats["statType"] == "ATT")) |
    ((qb_stats["category"] == "rushing") & (qb_stats["statType"] == "CAR"))
]

plays = attempts.pivot_table(index="player", columns="category",
                             values="stat", aggfunc="sum").reindex(qbs)
plays["total_plays"] = plays.sum(axis=1)
plays

category,passing,rushing,total_plays
player,,,
Darian Mensah,133.0,12.0,145.0
Dante Moore,99.0,10.0,109.0
Trinidad Chambliss,174.0,32.0,206.0
Sam Leavitt,173.0,60.0,233.0
Demond Williams Jr.,174.0,64.0,238.0
Josh Hoover,129.0,22.0,151.0
Drew Mestemaker,169.0,15.0,184.0
Jaron-Keawe Sagapolutele,164.0,13.0,177.0
Mason Heintschel,148.0,30.0,178.0


In [7]:
from pathlib import Path

logo_dir = Path("../cfb_logos")

r = requests.get("https://api.collegefootballdata.com/teams/fbs",
                 params={"year": 2026}, headers=headers)
r.raise_for_status()

for team in r.json():
    logos = team.get("logos") or []
    if logos:
        img = requests.get(logos[0])
        if img.ok:
            (logo_dir / f"{team['school']}.png").write_bytes(img.content)